In [10]:
import websocket
import json
import time
import random
import string
import requests
import time
from pprint import pprint

def random_id(length=17):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

# Step 1: get sockjs session info
info = requests.get("https://www.sboulder.com/sockjs/info").json()
print("Server info:", info)

# Step 2: build session URL
server_id = str(random.randint(0, 999))
session_id = random_id(8)
ws_url = f"wss://www.sboulder.com/sockjs/{server_id}/{session_id}/websocket"

received = []

boulders = {}
counters = {}
comments = {}
def on_message_test(ws, message):
    received.append(message)
    print("RECV:", message[:200])

def fetch_all_comments(ws, boulder_ids, delay=0.3):
    for bid in boulder_ids:
        sub_id = random_id()
        ws.send(json.dumps([json.dumps({
            "msg": "sub",
            "id": sub_id,
            "name": "_boulders.comments",
            "params": [bid]
        })]))
        time.sleep(delay)
comments_requested = False  # guard so we only do this once

def on_message(ws, message):
    global comments_requested
    if message == "o":
        return
    if message.startswith("a["):
        try:
            frames = json.loads(message[1:])
        except json.JSONDecodeError:
            return
        for frame in frames:
            try:
                data = json.loads(frame)
            except json.JSONDecodeError:
                continue

            msg_type = data.get("msg")

            if msg_type == "added" and data.get("collection") == "boulders":
                boulders[data["id"]] = data["fields"]

            elif msg_type == "changed" and data.get("collection") == "boulders":
                boulders.setdefault(data["id"], {}).update(data.get("fields", {}))

            elif msg_type == "added" and data.get("collection") == "comments":
                comments.setdefault(data["id"], data.get("fields", {}))

            elif msg_type == "ready":
                print("Subscription ready:", data.get("subs"))
                if len(boulders) > 0 and not comments_requested:
                    comments_requested = True
                    print(f"\n--- Collected {len(boulders)} boulders, fetching comments ---")
                    fetch_all_comments(ws, list(boulders.keys()))

def on_open(ws):
    print("Connected, sending DDP connect...")
    ws.send(json.dumps([json.dumps({
        "msg": "connect",
        "version": "1",
        "support": ["1", "pre2", "pre1"]
    })]))
    time.sleep(1)

    # Subscribe to boulders for a gym
    sub_id = random_id()
    ws.send(json.dumps([json.dumps({
        "msg": "sub",
        "id": sub_id,
        "name": "_boulders.list",
        "params": [
            {"gym": "arkose/montmartre", "isClosed": None},
            {"isClosed": 1, "createdAt": -1, "boulderNum": -1, "label": -1, "holdsColor": -1},
            200,
            None
        ]
    })]))

    # Subscribe to your personal sent count
    sub_id2 = random_id()
    # ws.send(json.dumps([json.dumps({
    #     "msg": "sub",
    #     "id": sub_id2,
    #     "name": "_boulders.count",
    #     "params": [{"gym": "arkose/montmartre", "sentsList": "qQFsxQKYvqRqYJNKa", "isClosed": None}]
    # })]))
    # ws.send(json.dumps([json.dumps({
    #     "msg": "sub",
    #     "id": sub_id2,
    #     "name":"_boulders.comments",
    #     "params":["arkose/montmartre"]
    # })]))


def on_error(ws, error):
    print("ERROR:", error)

def on_close(ws, code, msg):
    print("Closed:", code, msg)

ws = websocket.WebSocketApp(
    ws_url,
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close,
    header={"Origin": "https://www.sboulder.com"}
)

timer = 10
start = time.time()
current_time = start
ws.run_forever()
# while start < timer + start:
#     current_time = time.time()

Server info: {'websocket': True, 'origins': ['*:*'], 'cookie_needed': False, 'entropy': 2479864791}
Connected, sending DDP connect...
Subscription ready: ['iZB1nMhqlwhyzAjcI']

--- Collected 120 boulders, fetching comments ---
Subscription ready: ['YyDOqf85vvO5WElTN']
Subscription ready: ['6kn41jO1UMV0H5JZq']
Subscription ready: ['dgyJQaF0m5X2GQogY']
Subscription ready: ['ZIH8B3UQOvZS0EN5s']
Subscription ready: ['mNGnIVnAvsfChqlTQ']
Subscription ready: ['LPMBbmYTwz461J5FS']
Subscription ready: ['9Yx8Ts9ftixMffs3u']
Subscription ready: ['wFfYj9FufZSJCtpsv']
Subscription ready: ['vDhFVPtAxXVIKLv8K']
Subscription ready: ['oTG6X6ISKvDsk7mFy']
Subscription ready: ['etPFBFBB8mhTzC15k']
Subscription ready: ['JcEuRNUV2GtxcNkbl']
Subscription ready: ['Xt17LPmmqnqfyMwMl']
Subscription ready: ['VlC95YNZF05piqrcS']
Subscription ready: ['IpbDYLGWK2ZxlpN4c']
Subscription ready: ['A7ckgUgijzrwmdVtZ']
Subscription ready: ['kqzMW1MG3p8nkKZUQ']
Subscription ready: ['hfk8jCBJtrlVQbHDh']
Subscription read

True

In [12]:
comments

{'BkAYTfSfTZEytpag9': {'_isNew': False,
  'userId': 'BMqEfcEHBT3jSfigL',
  'boulderId': 'DhwmtYSEJsi5QZhJh',
  'boulder': {'gym': 'arkose/montmartre'},
  'userProfile': {'$type': 'Astronomy',
   '$value': {'class': 'UserProfile',
    'values': '{"name":"Clem Da Blocka","phones":[],"scores":{"arkose/montmartre":{"label":7}},"avatars":{"$type":"Astronomy","$value":{"class":"Avatars","values":"{\\"none\\":false,\\"uploaded\\":\\"45ozApJuTxq6i3M9c\\"}"}}}'}},
  'text': 'Pas la méthode attendue je pense mais ça marche.\nJ’arrivais pas à crawler main droite dans l’avant dernière donc switch main sous le volume et épaule gauche.',
  'videoId': 'UgEqWFSNEcOGivQ00JqQHHDD6W7URbFE7y02OPFGQQqaw',
  'videoSource': 'mux',
  'date': {'$date': 1785021754547},
  'gymInfos': {'name': 'Arkose Montmartre',
   'groupGyms': ['arkose'],
   'levelsGym': 'arkose',
   'appType': 'boulders'}},
 'jdjipj7G5KTJzoxDo': {'_isNew': False,
  'userId': 'YhmHgj6aKsAfPwJhN',
  'boulderId': 'y3G5TQMQh7RSAGD7F',
  'boulder'

In [ ]:
YOUR_ID = "qQFsxQKYvqRqYJNKa"

your_sends = []
for bid, b in boulders.items():
    if YOUR_ID in b.get("sentsList", []):
        # print(b)
        your_sends.append({
            "id": bid,
            "grade": b.get("grade"),
            "boulderNum": b.get("boulderNum"),
            "zone": b.get("zone"),
            "flashed": YOUR_ID in b.get("flashesList", []),
            "closedAt": b.get("closedAt"),
            "routeTypes": b.get("routeTypes")
        })

for s in your_sends:
    print(s)

Download the bundle to have the information about the hashtags of the different routes

In [ ]:
import requests
import re

url = "https://www.sboulder.com/76c5c9e4c16219899b2799d7b9c6d2584207a1a5.js"
js = requests.get(url).text


for m in re.finditer(r"displaySelectedRouteTypes", js):
    start = max(0, m.start() - 100)
    end = min(len(js), m.end() + 1000)
    print(js[start:end])
    print("===")
    